# Model Evaluation: Cross-Validation, ROC, and Bias-Variance

Choosing the right model requires rigorous evaluation. This notebook covers:
1. **Cross-validation** strategies
2. **ROC curves** and AUC
3. **Bias-variance trade-off** demonstration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    cross_val_score, StratifiedKFold, learning_curve, validation_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve, auc, RocCurveDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)

In [ ]:
# Load data
data = load_breast_cancer()
X, y = data.data, data.target
print(f"Shape: {X.shape}, Classes: {np.bincount(y)}")

## 1. Cross-Validation

**Stratified K-Fold** preserves class proportions in each fold -- essential for imbalanced data.

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
    print(f"{name:25s}  {scores.mean():.4f} +/- {scores.std():.4f}")

## 2. ROC Curve and AUC

The **Receiver Operating Characteristic** curve plots True Positive Rate vs False Positive Rate at varying thresholds. AUC = 1 is perfect; AUC = 0.5 is random.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

fig, ax = plt.subplots()
for name, model in models.items():
    model.fit(X_train, y_train)
    if hasattr(model, 'predict_proba'):
        y_scores = model.predict_proba(X_test)[:, 1]
    else:
        y_scores = model.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Bias-Variance Trade-off

- **High bias** (underfitting): model is too simple -- both train and test error are high.
- **High variance** (overfitting): model is too complex -- train error is low but test error is high.

We illustrate this with **learning curves** and **validation curves**.

In [ ]:
# Learning curve: how does performance change with more training data?
train_sizes, train_scores, test_scores = learning_curve(
    RandomForestClassifier(n_estimators=50, random_state=42),
    X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
)

plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train')
plt.plot(train_sizes, test_scores.mean(axis=1), 'o-', label='Validation')
plt.fill_between(train_sizes, train_scores.mean(1)-train_scores.std(1),
                 train_scores.mean(1)+train_scores.std(1), alpha=0.1)
plt.fill_between(train_sizes, test_scores.mean(1)-test_scores.std(1),
                 test_scores.mean(1)+test_scores.std(1), alpha=0.1)
plt.xlabel('Training set size')
plt.ylabel('Accuracy')
plt.title('Learning Curve (Random Forest)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Validation curve: how does tree depth affect bias/variance?
param_range = np.arange(1, 20)
train_s, test_s = validation_curve(
    DecisionTreeClassifier(random_state=42), X, y,
    param_name='max_depth', param_range=param_range,
    cv=5, scoring='accuracy'
)

plt.plot(param_range, train_s.mean(1), 'o-', label='Train')
plt.plot(param_range, test_s.mean(1), 'o-', label='Validation')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Validation Curve (Decision Tree)')
plt.legend()
plt.tight_layout()
plt.show()

print("Small depth = high bias (underfit), large depth = high variance (overfit)")

## Key Takeaways

- Always use **cross-validation** rather than a single train/test split.
- **ROC/AUC** gives a threshold-independent view of classifier quality.
- Use **learning curves** to diagnose whether more data would help.
- Use **validation curves** to tune hyperparameters and find the sweet spot between bias and variance.